## Summary

This notebook demonstrated how to work with custom acquisition functions in
Obsidian:

### What We Covered

1. **Exploring the Registry**
   - View registered functions by category
   - Inspect function configurations
   - Check defaults

2. **Creating Custom Functions**
   - Define a custom acquisition class inheriting from `MCAcquisitionFunction`
   - Implement the `forward()` method with proper shapes
   - Create a hyperparameter parser
   - Register with `acquisition_function_register()`

3. **Using Custom Functions**
   - Use custom functions just like built-ins
   - Pass custom hyperparameters
   - Compare with standard acquisition functions

4. **Modifying Internal Functions**
   - Override built-in implementations with `overloading=True`
   - Reuse original parsers with `reuse_parser=True`

5. **Resetting the Registry**
   - Restore defaults with `reset_registry()`
   - Remove custom functions and restore overridden ones

### Key Parameters for Registration

```python
acquisition_function_register(
    name="MyAcq",                     # Unique name, or if you want to overload an existing one (be careful with overloading!)
    implementation=MyClass,           # Your class
    hp_defaults={...},                # Default hyperparameters
    is_optimization=True,             # Task type
    is_characterization=False,        # Task type (placeholder for future use)
    is_single_target=True,            # Modality
    is_multi_target=False,            # Modality
    parser=my_parser,                 # Optional parser function (default is a dummy passthrough)
    overloading=False,                # Set True to override existing (Note that once you register an existing name, you cannot register it again without overloading=True)
    reuse_parser=False,               # Set True to keep existing parser (only works if overloading an existing function)
)
```

### Best Practices

- ✅ Test your custom functions thoroughly before using in production
- ✅ Use descriptive names that indicate the function's behavior
- ✅ Provide default hyperparameters that work reasonably well
- ✅ Document your acquisition function's expected behavior
- ⚠️ Be careful when overloading internal functions
- ⚠️ Reset the registry after experiments to avoid confusion or when things go
  wrong 

## 1. Exploring the Acquisition Function Registry

In [ ]:
from obsidian.acquisition import registry

# View all registered acquisition functions
print("All registered functions:")
print(list(registry.configs.keys()))

# View by category
print("\n=== By Category ===")
print(f"Single-objective optimization: {registry.valid_aqs['optimization']['single']}")
print(f"Multi-objective optimization: {registry.valid_aqs['optimization']['multi']}")

# View defaults
print("\n=== Default Functions ===")
print(f"Single-objective: {registry.aq_defaults['optimization']['single']}")
print(f"Multi-objective: {registry.aq_defaults['optimization']['multi']}")

# Inspect a specific function
print("\n=== Inspecting 'EI' ===")
ei_config = registry.get_config("EI")
print(f"Implementation: {ei_config.implementation.__name__}")
print(f"Hyperparameters: {ei_config.hyperparameter_defaults}")
print(f"Modalities: {ei_config.modalities}")
print(f"Task types: {ei_config.task_types}")

## 2. Creating a Custom Acquisition Function

Let's create a custom acquisition function that balances mean and standard
deviation with a tunable exploration weight.

### Step 1: Define the Acquisition Function Class

Your custom function should inherit from `MCAcquisitionFunction` from BoTorch.
You can refer to the
[BoTorch documentation](https://botorch.org/docs/tutorials/custom_acquisition/)
and the `obsidian` codebase for examples.

Note that the `obsidian` architecture by default requires a `constraints`
parameter for acquisition functions. If your implementation does not use or
support constraints, you can simply add the signature to init, but leave it
unused in the implementation.

In [ ]:
from botorch.acquisition import MCAcquisitionFunction
from botorch.utils.transforms import t_batch_mode_transform


class ExplorationWeightedMean(MCAcquisitionFunction):
    """Custom acquisition function: mean + exploration_weight * std"""

    def __init__(
        self,
        model,
        sampler=None,
        objective=None,
        posterior_transform=None,
        X_pending=None,
        exploration_weight=1.0,
        constraints=None,
        **kwargs
    ):
        super().__init__(model, sampler, objective, posterior_transform, X_pending)
        self.exploration_weight = exploration_weight

    @t_batch_mode_transform()
    def forward(self, X):
        """Compute acquisition value: mean + exploration_weight * std"""
        # X shape: [batch, q, d]
        posterior = self.model.posterior(X)
        samples = self.get_posterior_samples(posterior)  # [n_samples, batch, q, m]

        # Apply objective
        obj = self.objective(samples, X)  # [n_samples, batch, q] or [n_samples, batch, q, m]

        # Compute mean and std over samples
        mean = obj.mean(dim=0)  # [batch, q] or [batch, q, m]
        std = obj.std(dim=0)  # [batch, q] or [batch, q, m]

        # Combine with exploration weight
        acquisition_value = mean + self.exploration_weight * std

        # Sum over q (and m if multi-output)
        return (
            acquisition_value.sum(dim=-1).sum(dim=-1) if acquisition_value.dim() > 2 else acquisition_value.sum(dim=-1)
        )


print("Custom acquisition function class defined!")

### Step 2: Define a Hyperparameter Parser

A parser function processes hyperparameters before they are passed to your
acquisition function. It is needed if

1. Your acquisition function have required hyperparameters that do not have
   defaults, or
2. You want to make the optional variables configurable (which you should).

If you leave it empty, the registry will assign a dummy passthrough parser that
does nothing. Apparently, this could break your function if it does require any
processing.

```python
def default_hyperparameter_parser(
    aq_kwargs: dict[str, Any], hps: dict[str, Any], context: ParserContext | None = None
) -> dict[str, Any]:
    """Default dummy hyperparameter parser

    Args:
        aq_kwargs (dict[str, Any]): Acquisition function keyword arguments partially processed from the optimizer, including all default arguments of the acquisition function
        hps (dict[str, Any]): Hyperparameters passed in when `suggest` is called
        context (ParserContext | None, optional): Parser context, a typed dictionary, currently contains
        contextual information for hyperparameter parsing. Check its docstring for details. Defaults to None.

    Returns:
        dict[str, Any]: Parsed acquisition function keyword arguments
    """
    return aq_kwargs
```

All parsers should have the same signature as above. You can add any processing
logic you want, but make sure to return a dictionary of acquisition function
keyword arguments that match your acquisition function's `__init__` signature.

At this moment, the `context` object is defined as follows, 

```python
class ParserContext(TypedDict):
    """Context dictionary for acquisition function hyperparameter parsing.

    This typed dictionary provides contextual information to hyperparameter parsers,
    allowing an unified interface for parsing across different acquisition functions.

    Attributes:
        f_t: Transformed objective values for all observed data. Shape: (n_obs, n_targets).
            These are the transformed responses (via f_transform) for the target variables.

        X_baseline: Baseline input tensor containing all observed and pending points.
            Shape: (n_baseline, n_dim). Combines training data (X_train) and any pending
            evaluations (X_pending). Used by:
            - Noisy acquisition functions (NEI, NEHVI) for computing fantasies
            - Space-filling functions to avoid suggesting near-observed points
            - Objective transformations requiring reference to observed data

        m_batch: Number of candidates to propose in this batch. Used by acquisition
            functions that need to know batch size.

        n_dim: Dimensionality of the parameter space. Used for space-filling and other
            geometry-aware acquisition functions.

        target: List of Target objects describing the optimization objectives. Contains
            information about aim (max/min), transformation, tracking status, etc.

        objective: BoTorch MCAcquisitionObjective for transforming posterior samples.
            Used to scalarize multi-output models or apply custom transformations.
    """

    f_t: torch.Tensor
    X_baseline: torch.Tensor
    m_batch: int
    n_dim: int
    target: list[Target]
    objective: MCAcquisitionObjective | None
```
which should cover most use cases for acquisition function hyperparameter
parsing. We may expand it in the future to include more information about the
optimization state.

In [ ]:
def exploration_parser(aq_kwargs, hps, context):
    """
    Parse hyperparameters for ExplorationWeightedMean.
    
    Args:
        aq_kwargs: Keyword arguments that will be passed to the acquisition function
        hps: User-provided hyperparameters
        context: Additional context (e.g., training data, targets)
    
    Returns:
        Updated aq_kwargs dictionary
    """
    # Extract exploration_weight from hyperparameters
    aq_kwargs["exploration_weight"] = hps.get("exploration_weight", 1.0)
    return aq_kwargs

print("Parser function defined!")

### Step 3: Register the Function

Use `acquisition_function_register()` to add your function to the registry.

In [ ]:
from obsidian.acquisition import acquisition_function_register

acquisition_function_register(
    name="EWM",  # Short name for ExplorationWeightedMean
    implementation=ExplorationWeightedMean,
    hp_defaults={
        "exploration_weight": {"val": 1.0, "dtype": float, "optional": True}
    },
    is_optimization=True,      # For optimization tasks
    is_single_target=True,      # For single-objective problems
    is_multi_target=False,      # Not for multi-objective
    parser=exploration_parser,  # Our custom parser
)

print("✓ Custom acquisition function 'EWM' registered!")
print(f"✓ Now have {len(registry.configs)} functions in registry")
print(f"✓ 'EWM' available for: {registry.get_config('EWM').task_types}")

## 3. Using Custom Acquisition Functions

Now that our function is registered, we can use it just like any built-in acquisition function!

In [ ]:
import pandas as pd
from obsidian import BayesianOptimizer, Campaign, ParamSpace, Target
from obsidian.experiment import Simulator
from obsidian.experiment.benchmark import shifted_parab
from obsidian.parameters import Param_Continuous

params = [
    Param_Continuous("Temperature", -10, 30),
    Param_Continuous("Concentration", 10, 150),
    Param_Continuous("Enzyme", 0.01, 0.30),
]
X_space = ParamSpace(params)
target = Target("Yield", aim="max")
campaign = Campaign(X_space, target, seed=0)
X0 = campaign.initialize(m_initial=10, method="LHS")

simulator = Simulator(X_space, shifted_parab, name="Yield", eps=0.05)
y0 = simulator.simulate(X0)
Z0 = pd.concat([X0, y0], axis=1)
campaign.add_data(Z0)

In [ ]:
# Fit optimizer with training data
optimizer = BayesianOptimizer(X_space, seed=114514, verbose=0)
optimizer.fit(Z0, target=target)

# Use our custom acquisition function with default hyperparameters
X_suggest_default, eval_default = optimizer.suggest(
    m_batch=2, 
    acquisition=['EWM'],  # Our custom function!
)

print("=== Suggestions with default exploration_weight=1.0 ===")
print(X_suggest_default)

# Use with custom exploration weight (more exploration)
X_suggest_explore, eval_explore = optimizer.suggest(
    m_batch=2, 
    acquisition=[{'EWM': {'exploration_weight': 2.0}}],  # Higher exploration
    optim_samples=64,
    optim_restarts=4
)

print("\n=== Suggestions with exploration_weight=2.0 (more exploratory) ===")
print(X_suggest_explore)

# Compare with built-in NEI
X_suggest_nei, eval_nei = optimizer.suggest(
    m_batch=2,
    acquisition=['NEI'],  # Built-in function
)

print("\n=== Suggestions with NEI (built-in) ===")
print(X_suggest_nei)

## 4. Modifying Internal Functions (Advanced)

You can override built-in acquisition functions with custom implementations.
This is useful for experimenting with variations of standard methods.

⚠️ **Warning**: This is an advanced feature. Be careful when overriding internal functions!

In [ ]:
# Check the original implementation
print("Original UCB implementation:")
original_ucb = registry.get_config('UCB').implementation
print(f"  {original_ucb}")

# Override UCB with our custom implementation
acquisition_function_register(
    name="UCB",
    implementation=ExplorationWeightedMean,  # Use our custom class
    hp_defaults={
        "exploration_weight": {"val": 1.5, "dtype": float, "optional": True}
    },
    is_optimization=True,
    is_single_target=True,
    overloading=True,  # Must set this to True!
)

# Verify it was overloaded
print("\n✓ UCB has been overloaded!")
print(f"New implementation: {registry.get_config('UCB').implementation}")
print(f"Original != New: {original_ucb != registry.get_config('UCB').implementation}")

### Reusing Parsers When Overloading

When overloading, you can reuse the original function's parser with `reuse_parser=True`.

In [ ]:
# Get original EI parser
original_ei_parser = registry.get_config('EI').hyperparameter_parser
print(f"Original EI parser: {original_ei_parser.__name__}")

# Override implementation but keep the parser
class ModifiedEI(ExplorationWeightedMean):
    """Modified EI that uses our exploration strategy"""
    pass

acquisition_function_register(
    name="EI",
    implementation=ModifiedEI,
    is_optimization=True,
    is_single_target=True,
    overloading=True,
    reuse_parser=True,  # Keep original parser!
)

# Verify parser was reused
new_ei_parser = registry.get_config('EI').hyperparameter_parser
print(f"New EI parser: {new_ei_parser.__name__}")
print(f"✓ Parser reused: {original_ei_parser == new_ei_parser}")

## 5. Resetting the Registry

After experimenting with custom functions and overrides or if something is
really messed up, you can reset the registry to its default state.

In [ ]:
from obsidian.acquisition.config import reset_registry

print("Before reset:")
print(f"  Total functions: {len(registry.configs)}")
print(f"  'EWM' exists: {'EWM' in registry.configs}")
print(f"  UCB implementation: {registry.get_config('UCB').implementation.__name__}")

# Reset the registry
reset_registry()

print("\n✓ Registry has been reset!")
print(f"  Total functions: {len(registry.configs)}")
print(f"  'EWM' exists: {'EWM' in registry.configs}")
print(f"  UCB implementation: {registry.get_config('UCB').implementation.__name__}")

# Verify all built-in functions are restored
builtin_functions = ['EI', 'NEI', 'PI', 'UCB', 'SR', 'NIPV', 'EHVI', 'NEHVI', 'NParEGO', 'Mean', 'RS', 'SF']
all_present = all(name in registry.configs for name in builtin_functions)
print(f"  All built-in functions restored: {all_present}")